In [80]:
lines = '''\
aaaa
bbbbbb
cccccc
dddddddd'''

In [81]:
def get_lines(s):
    start = 0
    while (i := s.find('\n', start)) != -1:
        yield s[start:i]
        start = i+1
    yield s[start:]

In [82]:
list(get_lines(lines))

['aaaa', 'bbbbbb', 'cccccc', 'dddddddd']

In [ ]:
&uuml;

In [16]:
level = '''\
   #####\
####   #\
#    # #\
# $*.  #\
#  *@###\
##$*.#  \
 #   #  \
 #####  \
'''

In [58]:
def gen_indices(s, sub):
    i = 0
    while (i := s.find(sub, i)) > 0:
        yield i
        i += 1

In [59]:
border_idxs = list(gen_indices(level, '#'))
border_idxs[:2] + border[-2:]

[3, 4, 60, 61]

In [53]:
def find_chars(s, chars):
    for i, c in enumerate(s):
        if c in chars:
            yield i

In [54]:
box_idxs = list(find_chars(level, '$*'))
box_idxs

[26, 27, 35, 42, 43]

In [57]:
mat = [list(level[8*i:8*(i+1)]) for i in range(8)]
mat

[[' ', ' ', ' ', '#', '#', '#', '#', '#'],
 ['#', '#', '#', '#', ' ', ' ', ' ', '#'],
 ['#', ' ', ' ', ' ', ' ', '#', ' ', '#'],
 ['#', ' ', '$', '*', '.', ' ', ' ', '#'],
 ['#', ' ', ' ', '*', '@', '#', '#', '#'],
 ['#', '#', '$', '*', '.', '#', ' ', ' '],
 [' ', '#', ' ', ' ', ' ', '#', ' ', ' '],
 [' ', '#', '#', '#', '#', '#', ' ', ' ']]

In [63]:
def pos_and_values(mat, start_pos=(0, 0)):
    for row, values in enumerate(mat, start=start_pos[1]):
        for col, val in enumerate(values, start=start_pos[0]):
            pos = (col, row)
            yield pos, val


def get_positions(mat, chars, start_pos=(0, 0)):
    for pos, c in pos_and_values(mat, start_pos):
        if c in chars:
            yield pos

In [66]:
box_positions = list(get_positions(mat, '$*'))
box_positions

[(2, 3), (3, 3), (3, 4), (2, 5), (3, 5)]

In [67]:
box_positions == [divmod(i, 8)[::-1] for i in box_idxs]

True

 &uuml; &auml;

In [21]:
def numbers_generator(n):  # Generator-Funktion
    i = 1
    while i <= n:
        yield i
        i = i + 1

In [22]:
numbers_generator

<function __main__.numbers_generator(n)>

In [24]:
numbers = numbers_generator(3)
numbers

<generator object numbers_generator at 0x7f05624d2b50>

In [25]:
[next(numbers), next(numbers), next(numbers)]

[1, 2, 3]

In [26]:
next(numbers, 'I am done')

'I am done'

In [27]:
next(numbers)

StopIteration: 

In [28]:
squares = (x**2 for x in range(1, 10))
squares

<generator object <genexpr> at 0x7f0561dd1d80>

In [29]:
[next(squares) for _ in range(8)]

[1, 4, 9, 16, 25, 36, 49, 64]

In [30]:
list(squares)

[81]

In [31]:
next(squares, 'I am done')  

'I am done'

In [32]:
next(squares)

StopIteration: 

In [33]:
rg = range(3)  # kein Generator-Objekt, wiederverwendbar!
list(rg), list(rg)

([0, 1, 2], [0, 1, 2])

In [34]:
def fibonacci_gen():
    x, y = 1, 1
    while True:
        yield x
        x, y = y, x+y

In [35]:
fibonacci = fibonacci_gen()
fibonacci

<generator object fibonacci_gen at 0x7f0561dd20c0>

In [37]:
[next(fibonacci) for _ in range(10)]  # die 10 naechsten Fibonacci Zahlen

[1, 1, 2, 3, 5, 8, 13, 21, 34, 55]

In [ ]:
# list(fibonacci)  # runs forever!

In [44]:
def get_letters():
    char = 'A'
    while char <= 'Z':
        yield char
        char = chr(ord(char)+1)

In [45]:
letters = get_letters()

''.join(next(letters) for _ in range(13))

'ABCDEFGHIJKLM'

In [46]:
list(letters)

['N', 'O', 'P', 'Q', 'R', 'S', 'T', 'U', 'V', 'W', 'X', 'Y', 'Z']

In [47]:
next(letters, '.')

'.'

In [48]:
s = ''
for c in get_letters():
    s += c

s

'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

In [49]:
s = ''
letters = get_letters()
while True:
    try:
        c = next(letters)
    except StopIteration:
        break

    s += c

s

'ABCDEFGHIJKLMNOPQRSTUVWXYZ'

### Maze-Generation
Wir betrachten folgendes Spiel auf einem (m x n)-Gitter.
Jedes Gitterfeld ist von 4 Wänden eingeschlossen.  
Der Spieler startet auf Feld (0, 0). Das Spiel enden, wenn er alle Felder besucht hat.
- Der Spieler darf eine Wand einreissen, um auf
  ein noch **unbesuchtes Nachbarfeld** zu gelangen.
- Der Spieler kann auf ein besuchtes Feld zurückkehren, falls es dazu keine Wand einreissen muss. 


Benutzen eine Variante unseres Algorithmus zum Finden einer Zusammenhangskomponente, um
alle Felder zu besuchen. Wir speichern
alle besuchte Felder **mit noch intakten Wänden zu Nachbarfeldern** in einer Liste `stack`, und alle
**besuchten Felder** in einer Menge `visited`.

- Zu Beginn ist `stack = [(0, 0)]` und `visited = {(0, 0)}`.
- Solange noch Felder zu besuchen sind, machen wir folgendes:
  - entfernen das letzte Feld der Liste Stack
  - besuchen von diesem Feld aus, in zufälliger Reihenfolge, ein unbesuchtes Nachbarfeld und
    fügen diese Felder zu `visited` und `stack` hinzu.


In [3]:
import random


def get_random_neighbors(x, y, dims):
    directions = [(0, 1), (0, -1), (1, 0), (-1, 0)]
    random.shuffle(directions)

    for dx, dy in directions:
        nx, ny = x + dx, y + dy
        if 0 <= nx < dims[0] and 0 <= ny < dims[1]:
            yield (nx, ny)


def get_moves(start=(0, 0), dims=(4, 2)):
    stack = [(start)]
    visited = {start}

    while len(visited) < dims[0]*dims[1]:
        pos = stack.pop()
        for npos in get_random_neighbors(*pos, dims):
            if npos in visited:
                continue
            visited.add(npos)
            stack.append(npos)
            yield pos, npos

In [4]:
list(get_moves())

[((0, 0), (0, 1)),
 ((0, 0), (1, 0)),
 ((1, 0), (1, 1)),
 ((1, 0), (2, 0)),
 ((2, 0), (2, 1)),
 ((2, 0), (3, 0)),
 ((3, 0), (3, 1))]

In [13]:
import widget_helpers as W
import grid_helpers as G


def draw_maze(ncol, nrow, dx=20, dy=20):
    grid_spec = (0, 0, dx, dy, ncol, nrow)
    mcanvas = W.get_mcanvas(2, width=ncol*dx, height=nrow*dy)
    bg, fg = mcanvas

    G.draw_grid(bg, grid_spec)

    fg.stroke_style = 'blue'
    fg.line_width = 2
    for src, dst in get_moves(dims=(ncol, nrow)):
        p = G.cr2xy(*src, grid_spec, center=True)
        q = G.cr2xy(*dst, grid_spec, center=True)
        fg.stroke_line(*p, *q)
    display(mcanvas)

In [14]:
draw_maze(20, 10)

MultiCanvas(height=200, layout=Layout(border_bottom='1px solid black', border_left='1px solid black', border_r…